In [ ]:
import ipywidgets as widgets

num_nodes = widgets.IntSlider(min=5, max=30, value=10, step=1, description="#Nodes")
num_nodes

In [ ]:
num_edges = widgets.IntSlider(min=5, max=60, value=20, step=1, description="#Edges")
num_edges

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def random_graph(n: int, e: int) -> nx.Graph:
    return nx.gnm_random_graph(n, e)

graph1 = random_graph(num_nodes.value, num_edges.value)
print("First graph")
nx.draw(graph1, with_labels=True)
plt.show()

In [ ]:
is_isomorphic = widgets.Dropdown(
    options=[
        ("Second graph not isomorphic (honest prover)", False),
        ("Second graph isomorphic (lying prover)", True)],
    value=False,
    description="Scenario:",
)
is_isomorphic

In [ ]:
import random

def isomorphic_graph(graph: nx.Graph):
    nodes = list(graph.nodes())
    random.shuffle(nodes)
    mapping = {}

    for i in range(0, len(nodes)):
        mapping[i] = nodes.pop()
        
    print(mapping)
    
    return nx.relabel_nodes(graph1, mapping, copy=True)

def non_isomorphic_graph(first: nx.Graph) -> nx.Graph:
    assert num_edges.value < num_nodes.value * (num_nodes.value - 1) / 2, "Complete graph is always isomorphic"
    
    while True:
        second = random_graph(num_nodes.value, num_edges.value)
        if not nx.is_isomorphic(first, second):
            return second
    

if is_isomorphic.value:
    graph2 = isomorphic_graph(graph1)
    print("Second graph [isomorphic to first graph]")
else:
    graph2 = non_isomorphic_graph(graph1)
    print("Second graph [not isomorphic to first graph]")
   
    
nx.draw(graph2, with_labels=True)
plt.show()

In [ ]:
class ShuffleProver:
    def __init__(self, graph1: nx.Graph, graph2: nx.Graph):
        self.graphs = [graph1, graph2]
    
    def distinguish(self, shuffled_graph: nx.Graph) -> nx.Graph:
        # Index of graph in self.graphs that is isomorphic to shuffled_graph
        return next(index for index, graph in enumerate(self.graphs) if nx.is_isomorphic(graph, shuffled_graph))

class ShuffleVerifier:
    def __init__(self, graph1: nx.Graph, graph2: nx.Graph):
        self.graphs = [graph1, graph2]
    
    def random_shuffled_graph(self) -> nx.Graph:
        self.chosen_index = random.randrange(0, 2)
        chosen_graph = self.graphs[self.chosen_index]
        mapping = shuffle_nodes(chosen_graph)
        shuffled_graph = nx.relabel_nodes(chosen_graph, mapping, copy=True)
        
        return shuffled_graph
    
    def verify(self, index: int) -> bool:
        return index == self.chosen_index

In [ ]:
# Feel free to run this multiple times

prover = ShuffleProver(graph1, graph2)
verifier = ShuffleVerifier(graph1, graph2)
shuffled_graph = verifier.random_shuffled_graph()
index = prover.distinguish(shuffled_graph)

# Verifier is convinced (prover and verifier chose same graph)
if verifier.verify(index):
    # Two graphs were isomorphic (undistinguishable)
    if is_isomorphic.value:
        print("Convinced 👌 (verifier was fooled)")
    # Two graphs were non-isomorphic (distinguishable)
    else:
        print("Convinced 👌 (expected)")
# Verifier is not convinced (prover and verifier chose different graphs)
else:
    # Two graphs were isomorphic (undistinguishable)
    if is_isomorphic.value:
        print("Not convinced... 🤨 (expected)")
    # Two graphs were non-isomorphic (distinguishable)
    else:
        print("Not convinced... 🤨 (prover was dumb)")

In [ ]:
num_exchanges = widgets.IntSlider(min=10, max=1000, value=10, step=10, description="#Exchanges")
num_exchanges

In [ ]:
# completeness: honest prover has chance to win

graph3 = non_isomorphic_graph(graph1)
honest_prover = ShuffleProver(graph1, graph3)
verifier = ShuffleVerifier(graph1, graph3)

prover_success = 0

for _ in range(0, num_exchanges.value):
    shuffled_graph = verifier.random_shuffled_graph()
    index = honest_prover.distinguish(shuffled_graph)
    
    if verifier.verify(index):
        prover_success += 1

print("Running {} exchanges".format(num_exchanges.value))
print("Honest prover wins with probability {0:0.8f}".format(prover_success / num_exchanges.value))

In [ ]:
# Boost verifier confidence to 1 - 0.5^i by running i times
num_rounds = widgets.IntSlider(min=1, max=10, value=1, step=1, description="#Rounds")
num_rounds

In [ ]:
# soundness: verifier has chance to win against lying prover

mapping = shuffle_nodes(graph1)
graph4 = nx.relabel_nodes(graph1, mapping, copy=True)
lying_prover = ShuffleProver(graph1, graph4)
verifier = ShuffleVerifier(graph1, graph4)

verifier_success = 0

for _ in range(0, num_exchanges.value):
    for _ in range(0, num_rounds.value):
        shuffled_graph = verifier.random_shuffled_graph()
        index = honest_prover.distinguish(shuffled_graph)
    
        if not verifier.verify(index):
            verifier_success += 1
            break

print("Running {} exchanges with {} rounds each"
      .format(num_exchanges.value, num_rounds.value))
print("Verifier wins against lying prover with probability {0:0.8f}"
      .format(verifier_success / num_exchanges.value))

In [ ]:
num_transcripts = widgets.IntSlider(min=1000, max=20000, value=1000, step=1000, description="#Transcripts")
num_transcripts

In [ ]:
from typing import List, Tuple

#todo: zero-knowledge
# number of edges should be small for a clear distribution to form after few iterations

prover = ShuffleProver(graph1, graph2)

def real_transcript() -> Tuple[nx.Graph, int]:
    verifier = ShuffleVerifier(graph1, graph2)
    shuffled_graph = verifier.random_shuffled_graph()
    index = honest_prover.distinguish(shuffled_graph)
    return (shuffled_graph, index)

# For all elements el in lst: 0 <= el < bound
def compress_list(lst: List[int], bound: int) -> int:
    acc = 0
    for (i, el) in enumerate(lst):
        acc += el * bound ** i
    return acc

def compress(shuffled_graph: nx.Graph, index: int) -> int:
    edges = [x for edge in shuffled_graph.edges() for x in edge]
    acc = compress_list(edges, num_nodes.value)
    acc += index * num_nodes.value ** len(edges)
    return acc

real_data = []

for _ in range(0, num_transcripts.value):
    real_data.append(compress(*real_transcript()))

# there are mountains and valleys because graph1 and graph2 are constant
# the distribution should look different for different graphs
plt.hist(real_data)
plt.show()

In [ ]:
def fake_transcript():
    index = random.randrange(0, 2)
    chosen_graph = [graph1, graph2][index]
    mapping = shuffle_nodes(chosen_graph)
    shuffled_graph = nx.relabel_nodes(chosen_graph, mapping, copy=True)
    return shuffled_graph, index

fake_data = []

for _ in range(0, num_transcripts.value):
    fake_data.append(compress(*fake_transcript()))
    
plt.hist(fake_data)
plt.show()